In [4]:
# -----------------------------------------------------------------------------
# CS 340 Client/Server Development
# Project Two - Grazioso Salvare Dashboard
#
# Developer: Justin Carlo Florida
#
# This dashboard connects to the Austin Animal Center MongoDB database through
# the AnimalShelter CRUD Python module. The application allows Grazioso Salvare
# users to filter dogs by rescue type and dynamically updates the data table,
# breed distribution chart, and geolocation map based on the selected filter.
# -----------------------------------------------------------------------------

# Setup the Jupyter version of Dash.
from jupyter_dash import JupyterDash

# Import Dash components used to build the dashboard interface.
import dash_leaflet as dl
from dash import dcc, html
from dash import dash_table
from dash.dependencies import Input, Output

# Import Plotly Express for the dashboard chart.
import plotly.express as px

# Import supporting Python libraries.
import base64
import pandas as pd

# Import the AnimalShelter CRUD class used to communicate with MongoDB.
from CRUD_Python_Module import AnimalShelter


# Configure JupyterDash for the Codio/Jupyter environment.
JupyterDash.infer_jupyter_proxy_config()


# =============================================================================
# Data Manipulation / Model
# =============================================================================

# MongoDB credentials for the application user.
username = "aacuser"
password = "passwordPrompt()"

# Create an instance of the AnimalShelter CRUD class.
# This object provides the connection between the dashboard and MongoDB.
db = AnimalShelter(username, password)

# Retrieve all animal records to initialize the dashboard.
# An empty MongoDB query {} returns the unfiltered collection.
df = pd.DataFrame.from_records(db.read({}))

# MongoDB returns an ObjectId in the "_id" field. Dash cannot serialize the
# ObjectId into JSON, so remove the column if it is present.
if "_id" in df.columns:
    df.drop(columns=["_id"], inplace=True)


# =============================================================================
# Dashboard Layout / View
# =============================================================================

# Create the JupyterDash application.
app = JupyterDash(__name__)


# -------------------------------------------------------------------------
# Grazioso Salvare Logo
# -------------------------------------------------------------------------

image_filename = "Grazioso Salvare Logo.png"

# Convert the logo image to Base64 so it can be displayed in the dashboard.
encoded_image = base64.b64encode(
    open(image_filename, "rb").read()
).decode()


# -------------------------------------------------------------------------
# Dashboard User Interface
# -------------------------------------------------------------------------

app.layout = html.Div([

    # Display the clickable Grazioso Salvare logo, dashboard title,
    # and developer identifier at the top of the page.
    html.Center([

        # The Dashboard Specifications Document requires the Grazioso
        # Salvare logo to link to the client's home page.
        html.A(
            html.Img(
                src="data:image/png;base64,{}".format(encoded_image),
                style={
                    "height": "120px",
                    "marginBottom": "10px"
                }
            ),
            href="https://www.snhu.edu",
            target="_blank"
        ),

        # Dashboard title.
        html.H1("Grazioso Salvare Rescue Dashboard"),

        # Unique developer identifier required by Project Two.
        html.H4("Developer: Justin Carlo Florida")
    ]),

    html.Hr(),

    # ---------------------------------------------------------------------
    # Rescue Type Filtering Options
    # ---------------------------------------------------------------------

    html.Div([
        html.H3("Select Rescue Type:"),

        # RadioItems provides four intuitive options for filtering the
        # dashboard according to Grazioso Salvare's rescue requirements.
        dcc.RadioItems(
            id="filter-type",

            options=[
                {
                    "label": "Water Rescue",
                    "value": "water"
                },
                {
                    "label": "Mountain or Wilderness Rescue",
                    "value": "wilderness"
                },
                {
                    "label": "Disaster or Individual Tracking",
                    "value": "disaster"
                },
                {
                    "label": "Reset",
                    "value": "reset"
                }
            ],

            # Reset is the default option so the dashboard initially shows
            # the complete, unfiltered Austin Animal Center data set.
            value="reset",

            # Display the radio buttons horizontally for easier navigation.
            labelStyle={
                "display": "inline-block",
                "marginRight": "25px"
            },

            style={
                "marginBottom": "15px"
            }
        )
    ]),

    html.Hr(),

    # ---------------------------------------------------------------------
    # Interactive Data Table
    # ---------------------------------------------------------------------

    dash_table.DataTable(
        id="datatable-id",

        # Create the table columns from the MongoDB data fields.
        columns=[
            {
                "name": column,
                "id": column,
                "deletable": False,
                "selectable": True
            }
            for column in df.columns
        ],

        # Initially display the complete, unfiltered data set.
        data=df.to_dict("records"),

        # Allow only one animal to be selected at a time because the
        # geolocation map displays one selected animal.
        row_selectable="single",

        # Select the first row by default so the map has an animal to display.
        selected_rows=[0],

        # Allow users to sort columns directly in the dashboard.
        sort_action="native",

        # Allow users to perform additional native DataTable filtering.
        filter_action="native",

        # Use native pagination to keep the table manageable.
        page_action="native",
        page_size=10,

        # Allow horizontal scrolling when the table contains many columns.
        style_table={
            "overflowX": "auto"
        },

        # Improve readability of individual table cells.
        style_cell={
            "textAlign": "left",
            "minWidth": "100px",
            "width": "150px",
            "maxWidth": "200px",
            "whiteSpace": "normal"
        },

        # Make the column headers visually distinct.
        style_header={
            "fontWeight": "bold",
            "textAlign": "center"
        }
    ),

    html.Br(),
    html.Hr(),

    # ---------------------------------------------------------------------
    # Chart and Geolocation Map
    # ---------------------------------------------------------------------
    # The two dashboard visualizations are displayed side-by-side.

    html.Div(
        className="row",

        style={
            "display": "flex",
            "gap": "20px"
        },

        children=[

            # Plotly chart container.
            html.Div(
                id="graph-id",
                className="col s12 m6",
                style={
                    "width": "50%"
                }
            ),

            # Dash Leaflet geolocation map container.
            html.Div(
                id="map-id",
                className="col s12 m6",
                style={
                    "width": "50%"
                }
            )
        ]
    )
])


# =============================================================================
# Interaction Between Components / Controller
# =============================================================================


# -------------------------------------------------------------------------
# Rescue Type Filter Callback
# -------------------------------------------------------------------------
# This callback receives the rescue option selected by the user, constructs
# the corresponding MongoDB query, sends the query through the AnimalShelter
# CRUD module, and returns the matching records to the DataTable.
#
# MongoDB performs the filtering. The complete collection is NOT retrieved
# first and then filtered with Pandas/Python.
# -------------------------------------------------------------------------

@app.callback(
    Output("datatable-id", "data"),
    [Input("filter-type", "value")]
)
def update_dashboard(filter_type):
    """Return MongoDB records matching the selected rescue category."""

    # ---------------------------------------------------------------------
    # Water Rescue
    # ---------------------------------------------------------------------
    # Preferred breeds:
    #   Labrador Retriever Mix
    #   Chesapeake Bay Retriever
    #   Newfoundland
    #
    # Preferred sex:
    #   Intact Female
    #
    # Training age:
    #   26 through 156 weeks
    if filter_type == "water":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Labrador Retriever Mix",
                    "Chesapeake Bay Retriever",
                    "Newfoundland"
                ]
            },

            "sex_upon_outcome": "Intact Female",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # ---------------------------------------------------------------------
    # Mountain or Wilderness Rescue
    # ---------------------------------------------------------------------
    # Preferred breeds:
    #   German Shepherd
    #   Alaskan Malamute
    #   Old English Sheepdog
    #   Siberian Husky
    #   Rottweiler
    #
    # Preferred sex:
    #   Intact Male
    #
    # Training age:
    #   26 through 156 weeks
    elif filter_type == "wilderness":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "German Shepherd",
                    "Alaskan Malamute",
                    "Old English Sheepdog",
                    "Siberian Husky",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 26,
                "$lte": 156
            }
        }

    # ---------------------------------------------------------------------
    # Disaster or Individual Tracking
    # ---------------------------------------------------------------------
    # Preferred breeds:
    #   Doberman Pinscher
    #   German Shepherd
    #   Golden Retriever
    #   Bloodhound
    #   Rottweiler
    #
    # Preferred sex:
    #   Intact Male
    #
    # Training age:
    #   20 through 300 weeks
    elif filter_type == "disaster":

        query = {
            "animal_type": "Dog",

            "breed": {
                "$in": [
                    "Doberman Pinscher",
                    "German Shepherd",
                    "Golden Retriever",
                    "Bloodhound",
                    "Rottweiler"
                ]
            },

            "sex_upon_outcome": "Intact Male",

            "age_upon_outcome_in_weeks": {
                "$gte": 20,
                "$lte": 300
            }
        }

    # ---------------------------------------------------------------------
    # Reset
    # ---------------------------------------------------------------------
    # An empty query returns all records and restores the dashboard to its
    # original, unfiltered state.
    else:
        query = {}

    # Send the rescue query through the CRUD module to MongoDB.
    results = db.read(query)

    # Convert the returned MongoDB documents into a local DataFrame.
    # A local variable is used so the global DataFrame is not overwritten.
    dff = pd.DataFrame.from_records(results)

    # Remove the MongoDB ObjectId before converting the data to JSON records.
    if "_id" in dff.columns:
        dff.drop(columns=["_id"], inplace=True)

    # Convert the DataFrame to a list of dictionaries required by
    # the Dash DataTable data property.
    return dff.to_dict("records")


# -------------------------------------------------------------------------
# Dynamic Breed Distribution Chart
# -------------------------------------------------------------------------
# This callback receives the DataTable's CURRENT visible data rather than the
# original global DataFrame. Therefore, the chart automatically changes when
# the rescue filter changes or when the user uses DataTable filtering.
# -------------------------------------------------------------------------

@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")]
)
def update_graphs(viewData):
    """Create a pie chart from the animals currently displayed in the table."""

    # Prevent the chart callback from failing if no animals match a filter.
    if not viewData:
        return [
            html.Div([
                html.H3("Breed Distribution"),
                html.P("No animals match the current filter.")
            ])
        ]

    # Convert the current DataTable records into a DataFrame.
    dff = pd.DataFrame.from_dict(viewData)

    # Create a pie chart that shows the distribution of breeds among the
    # animals currently available in the filtered DataTable.
    figure = px.pie(
        dff,
        names="breed",
        title="Breed Distribution for Selected Rescue Type"
    )

    # Return the Plotly graph to the dashboard.
    return [
        dcc.Graph(
            figure=figure
        )
    ]


# -------------------------------------------------------------------------
# DataTable Selected Column Styling
# -------------------------------------------------------------------------
# Highlight a column when the user selects its header.
# -------------------------------------------------------------------------

@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")]
)
def update_styles(selected_columns):
    """Highlight selected DataTable columns."""

    # selected_columns can be empty, so return no styling in that case.
    if not selected_columns:
        return []

    return [
        {
            "if": {
                "column_id": column
            },
            "background_color": "#D2F3FF"
        }
        for column in selected_columns
    ]


# -------------------------------------------------------------------------
# Dynamic Geolocation Map
# -------------------------------------------------------------------------
# derived_virtual_data represents the records currently displayed in the
# DataTable after rescue filters, native table filters, and sorting.
#
# derived_virtual_selected_rows contains the currently selected row index.
# Because row_selectable="single", only one row can be selected at a time.
# -------------------------------------------------------------------------

@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows")
    ]
)
def update_map(viewData, index):
    """Display the location of the selected animal on a Dash Leaflet map."""

    # Do not try to create a marker if the DataTable contains no records.
    if not viewData:
        return [
            html.Div([
                html.H3("Animal Location"),
                html.P("No animal location is available for this filter.")
            ])
        ]

    # Convert the current DataTable data into a DataFrame.
    dff = pd.DataFrame.from_dict(viewData)

    # selected_rows may be an empty list rather than None.
    # When nothing has been manually selected, use the first visible row.
    if not index:
        row = 0
    else:
        row = index[0]

    # Make sure the selected row remains within the current filtered data.
    # This can occur when the rescue filter changes after a row was selected.
    if row >= len(dff):
        row = 0

    # Retrieve the latitude and longitude for the selected animal.
    latitude = dff.iloc[row]["location_lat"]
    longitude = dff.iloc[row]["location_long"]

    # Retrieve descriptive information for the marker.
    animal_name = dff.iloc[row]["name"]
    animal_breed = dff.iloc[row]["breed"]
    animal_id = dff.iloc[row]["animal_id"]

    # Center the map on the Austin, Texas region and place a marker at the
    # selected animal's shelter location.
    return [
        dl.Map(
            style={
                "width": "100%",
                "height": "500px"
            },

            center=[30.75, -97.48],
            zoom=10,

            children=[

                # OpenStreetMap base map layer.
                dl.TileLayer(
                    id="base-layer-id"
                ),

                # Marker representing the currently selected animal.
                dl.Marker(
                    position=[
                        latitude,
                        longitude
                    ],

                    children=[

                        # Show the breed when the user hovers over the marker.
                        dl.Tooltip(
                            animal_breed
                        ),

                        # Show additional animal information when the marker
                        # is selected.
                        dl.Popup([
                            html.H3("Animal Information"),
                            html.P(
                                "Animal ID: {}".format(animal_id)
                            ),
                            html.P(
                                "Name: {}".format(animal_name)
                            ),
                            html.P(
                                "Breed: {}".format(animal_breed)
                            )
                        ])
                    ]
                )
            ]
        )
    ]


# =============================================================================
# Run Dashboard
# =============================================================================
# Run the Dash application inside the Jupyter/Codio environment.
# If port 8050 is unavailable because another dashboard is already running,
# stop the previous server or restart the Jupyter kernel before running again.

app.run_server()

Dash app running on https://chrisboxer-majorparadox-3000.codio.io/proxy/8050/
